# Aerial Human Detection — Three-Model Inference + ByteTrack
## Google Colab — YOLO26s / RT-DETR-R18 / BPD-YOLOn/L-FPN

This notebook runs the three final project detectors on a user-supplied image or video.

**Behavior**
- **Image:** detection only. The original image is preserved and a technical information panel is appended **below** it.
- **Video:** detection + **ByteTrack**. The video includes boxes, person IDs, confidence, trails, source FPS, measured processing FPS, detector latency, total frame latency, counts, and the Step-3 validation reference metrics.
- `RUN_MODEL="ALL"` runs the same media sequentially with all three models and saves a compact runtime comparison CSV.

**Important interpretation**
- `Mean Det Conf` is the detector's mean confidence on the current media. It is **not** ground-truth accuracy.
- `Step3 Val mAP50-95` and `AP50` are the already measured VisDrone validation metrics and are displayed as reference values.
- HOTA, IDF1, MOTA and ID-switch accuracy require annotated tracking ground truth and are not fabricated here.

All outputs are written to Google Drive.


## 1. Mount Google Drive


In [25]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
print("Google Drive mounted.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted.


## 2. Install controlled inference dependencies


In [26]:
import subprocess
import sys

packages = [
    "ultralytics==8.4.116",
    "PyYAML>=6.0",
    "pandas>=2.0",
    "tqdm>=4.66",
    "pillow>=10.0",
]

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade-strategy",
        "only-if-needed",
        *packages,
    ]
)

import cv2
import numpy as np
import pandas as pd
import torch
import torchvision
import ultralytics

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Torchvision :", torchvision.__version__)
print("CUDA        :", torch.version.cuda)
print("Ultralytics :", ultralytics.__version__)
print("OpenCV      :", cv2.__version__)

if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA is unavailable. Inference will run on CPU.")

print("Runtime setup: PASSED")


Python      : 3.12.13
PyTorch     : 2.11.0+cu128
Torchvision : 0.26.0+cu128
CUDA        : 12.8
Ultralytics : 8.4.116
OpenCV      : 5.0.0
GPU         : Tesla T4
Runtime setup: PASSED


## 3. USER CONFIGURATION — edit only this cell

Paste the three final checkpoint paths from Google Drive and the image/video path.

`RUN_MODEL` may be:
- `"YOLO26s"`
- `"RT-DETR-R18"`
- `"BPD-YOLOn/L-FPN"`
- `"ALL"`

The common detector profile is intentionally explicit. ByteTrack values are also exposed so they can be changed if the GUI uses different tracker thresholds.


In [27]:
from pathlib import Path
import torch

# ============================================================
# FINAL MODEL PATHS IN GOOGLE DRIVE
# ============================================================

YOLO26S_MODEL_PATH = Path(
    "https://drive.google.com/file/d/1Ds2wuTMp1voGOH-VbNhqFfM-xwhpvBcL/view?usp=drive_link"
)

RTDETR_MODEL_PATH = Path(
    "https://drive.google.com/file/d/1Ku_Sh9i-aCN2aLaDQH8tDfgrTzBFzZ3M/view?usp=drive_link"
)

BPD_MODEL_PATH = Path(
    "https://drive.google.com/file/d/1sV5nb8uU4E_b_7JDitpbYbxoM0x2ob2H/view?usp=drive_link"
)


# ============================================================
# IMAGE OR VIDEO PATH
# ============================================================

MEDIA_PATH = Path(
    "https://drive.google.com/drive/folders/1Gv5E3ju5fg2nC9C2GNg7IHPeNbPsuuKD?usp=drive_link"
)


# ============================================================
# MODEL SELECTION
# ============================================================

RUN_MODEL = "YOLO26s"
# RUN_MODEL = "RT-DETR-R18"
# RUN_MODEL = "BPD-YOLOn/L-FPN"
# RUN_MODEL = "ALL"


# ============================================================
# OUTPUT
# ============================================================

OUTPUT_ROOT = Path(
    "https://drive.google.com/file/d/15KKiS3emSRCEOKswk_EZfED7iZSvpbmq/view?usp=drive_link"
)


# ============================================================
# COMMON DETECTOR SETTINGS
# ============================================================

IMAGE_SIZE = 1280
CONF_THRESHOLD = 0.10
IOU_THRESHOLD = 0.70
MAX_DETECTIONS = 3000
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# Keep False for reference PyTorch inference.
# Enable only when you intentionally want an FP16 inference experiment.
USE_FP16 = False


# ============================================================
# BYTETRACK SETTINGS
# ============================================================

TRACK_HIGH_THRESH = 0.25
TRACK_LOW_THRESH = 0.10
NEW_TRACK_THRESH = 0.25
TRACK_BUFFER = 30
MATCH_THRESH = 0.80
FUSE_SCORE = True


# ============================================================
# DISPLAY / EXPORT SETTINGS
# ============================================================

SHOW_BOXES = True
SHOW_TRACK_ID = True
SHOW_CONFIDENCE = True
SHOW_TRAILS = True
TRAIL_LENGTH = 30

SAVE_VIDEO = True
SAVE_CSV = True
SAVE_JSON = True

VIDEO_FRAME_STRIDE = 1

BOX_THICKNESS = 2
TEXT_SCALE = 0.55
TEXT_THICKNESS = 1

WARMUP_RUNS = 2
FPS_SMOOTHING_WINDOW = 30


# ============================================================
# PINNED RT-DETR SOURCE USED BY THIS PROJECT
# ============================================================

RTDETR_REPO_URL = "https://github.com/lyuwenyu/RT-DETR.git"
RTDETR_REPO_COMMIT = "199fc382f53abbfb5c1804c97b0e8b204e3cb8d0"


# ============================================================
# STEP-3 VALIDATION REFERENCE METRICS
# These are NOT new-media ground-truth accuracy.
# ============================================================

STEP3_VALIDATION = {
    "YOLO26s": {
        "map50_95": 0.31060,
        "ap50": 0.67438,
        "best_epoch": 30,
    },
    "RT-DETR-R18": {
        "map50_95": 0.34854,
        "ap50": 0.70839,
        "best_epoch": 45,
    },
    "BPD-YOLOn/L-FPN": {
        "map50_95": 0.29366,
        "ap50": 0.65557,
        "best_epoch": 43,
    },
}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Run model   :", RUN_MODEL)
print("Media       :", MEDIA_PATH)
print("Device      :", DEVICE)
print("Image size  :", IMAGE_SIZE)
print("Confidence  :", CONF_THRESHOLD)
print("IoU         :", IOU_THRESHOLD)
print("Max det     :", MAX_DETECTIONS)
print("Output root :", OUTPUT_ROOT)


Run model   : YOLO26s
Media       : https:/drive.google.com/drive/folders/1Gv5E3ju5fg2nC9C2GNg7IHPeNbPsuuKD?usp=drive_link
Device      : cuda:0
Image size  : 1280
Confidence  : 0.1
IoU         : 0.7
Max det     : 3000
Output root : https:/drive.google.com/file/d/15KKiS3emSRCEOKswk_EZfED7iZSvpbmq/view?usp=drive_link


## 4. Shared imports, structures, metrics and drawing utilities


In [28]:
from __future__ import annotations

import contextlib
import gc
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import time

from collections import defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace

import cv2
import numpy as np
import pandas as pd
import torch

from PIL import Image
from IPython.display import Video, display
from tqdm.auto import tqdm


IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"
}

VIDEO_EXTENSIONS = {
    ".mp4", ".mov", ".avi", ".mkv", ".m4v", ".webm",
    ".wmv", ".mpeg", ".mpg"
}


@dataclass
class DetectionBatch:
    boxes: np.ndarray
    scores: np.ndarray
    classes: np.ndarray
    detector_ms: float

    @classmethod
    def empty(cls, detector_ms: float = 0.0):
        return cls(
            boxes=np.zeros((0, 4), dtype=np.float32),
            scores=np.zeros((0,), dtype=np.float32),
            classes=np.zeros((0,), dtype=np.float32),
            detector_ms=float(detector_ms),
        )

    def __len__(self):
        return len(self.boxes)


def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as stream:
        while True:
            block = stream.read(block_size)
            if not block:
                break
            digest.update(block)

    return digest.hexdigest()


def sanitize_name(value: str) -> str:
    value = value.replace("/", "_").replace("\\", "_")
    return "".join(
        ch if ch.isalnum() or ch in {"-", "_"} else "_"
        for ch in value
    )


def classify_media(path: Path) -> str:
    suffix = path.suffix.lower()

    if suffix in IMAGE_EXTENSIONS:
        return "image"

    if suffix in VIDEO_EXTENSIONS:
        return "video"

    image = cv2.imread(str(path))
    if image is not None:
        return "image"

    cap = cv2.VideoCapture(str(path))
    ok = cap.isOpened()
    cap.release()

    if ok:
        return "video"

    raise ValueError(
        f"Could not classify the media as image or video: {path}"
    )


def deterministic_track_color(track_id: int):
    hue = int((track_id * 47) % 180)
    hsv = np.uint8([[[hue, 210, 255]]])
    bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)[0, 0]
    return tuple(int(x) for x in bgr.tolist())


def draw_text_box(
    image,
    text,
    origin,
    bg_color=(20, 20, 20),
    fg_color=(255, 255, 255),
    alpha=0.78,
    scale=0.55,
    thickness=1,
):
    x, y = origin

    (tw, th), baseline = cv2.getTextSize(
        text,
        cv2.FONT_HERSHEY_SIMPLEX,
        scale,
        thickness,
    )

    x1 = max(0, int(x))
    y1 = max(0, int(y - th - baseline - 6))
    x2 = min(image.shape[1] - 1, int(x + tw + 8))
    y2 = min(image.shape[0] - 1, int(y + 2))

    overlay = image.copy()
    cv2.rectangle(overlay, (x1, y1), (x2, y2), bg_color, -1)
    cv2.addWeighted(overlay, alpha, image, 1.0 - alpha, 0, image)

    cv2.putText(
        image,
        text,
        (x1 + 4, int(y - baseline - 2)),
        cv2.FONT_HERSHEY_SIMPLEX,
        scale,
        fg_color,
        thickness,
        cv2.LINE_AA,
    )


def draw_detection_boxes(
    frame,
    boxes,
    scores,
    track_ids=None,
    trails=None,
):
    output = frame.copy()

    if track_ids is None:
        track_ids = [None] * len(boxes)

    for box, score, track_id in zip(boxes, scores, track_ids):
        x1, y1, x2, y2 = [int(round(v)) for v in box]

        color = (
            deterministic_track_color(int(track_id))
            if track_id is not None
            else (0, 200, 255)
        )

        if SHOW_BOXES:
            cv2.rectangle(
                output,
                (x1, y1),
                (x2, y2),
                color,
                BOX_THICKNESS,
            )

        label_parts = ["person"]

        if SHOW_TRACK_ID and track_id is not None:
            label_parts.append(f"ID {int(track_id)}")

        if SHOW_CONFIDENCE:
            label_parts.append(f"{float(score):.2f}")

        draw_text_box(
            output,
            " | ".join(label_parts),
            (x1, max(18, y1)),
            bg_color=color,
            fg_color=(255, 255, 255),
            scale=TEXT_SCALE,
            thickness=TEXT_THICKNESS,
        )

    if SHOW_TRAILS and trails:
        for track_id, points in trails.items():
            if len(points) < 2:
                continue

            pts = np.asarray(points, dtype=np.int32).reshape(-1, 1, 2)

            cv2.polylines(
                output,
                [pts],
                False,
                deterministic_track_color(int(track_id)),
                2,
                cv2.LINE_AA,
            )

    return output


def add_video_status_overlay(
    frame,
    model_name,
    frame_number,
    source_fps,
    processing_fps,
    detector_ms,
    total_ms,
    detections,
    active_tracks,
    unique_tracks,
    mean_conf,
):
    output = frame.copy()
    panel_h = min(118, max(92, int(output.shape[0] * 0.11)))

    overlay = output.copy()
    cv2.rectangle(
        overlay,
        (0, 0),
        (output.shape[1], panel_h),
        (10, 16, 24),
        -1,
    )

    cv2.addWeighted(overlay, 0.78, output, 0.22, 0, output)

    ref = STEP3_VALIDATION[model_name]

    lines = [
        (
            f"{model_name} | person | ByteTrack | Frame {frame_number} | "
            f"Source {source_fps:.2f} FPS"
        ),
        (
            f"Processing {processing_fps:.2f} FPS | Detector {detector_ms:.1f} ms | "
            f"Total {total_ms:.1f} ms | Det {detections} | "
            f"Active IDs {active_tracks} | Unique IDs {unique_tracks}"
        ),
        (
            f"Mean Det Conf {mean_conf:.3f} | Step3 Val mAP50-95 "
            f"{ref['map50_95']:.5f} | AP50 {ref['ap50']:.5f} | "
            f"imgsz {IMAGE_SIZE} | conf {CONF_THRESHOLD:.2f}"
        ),
    ]

    y = 28

    for line in lines:
        cv2.putText(
            output,
            line,
            (14, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.58,
            (255, 255, 255),
            1,
            cv2.LINE_AA,
        )
        y += 30

    return output


def build_image_footer(
    annotated,
    model_name,
    model_path,
    detector_ms,
    detections,
    mean_conf,
):
    _, width = annotated.shape[:2]
    footer_h = 190

    footer = np.full(
        (footer_h, width, 3),
        (22, 28, 36),
        dtype=np.uint8,
    )

    ref = STEP3_VALIDATION[model_name]
    fps = 1000.0 / detector_ms if detector_ms > 0 else 0.0

    lines = [
        (
            f"Model: {model_name} | Class: person | "
            f"Checkpoint: {Path(model_path).name}"
        ),
        (
            f"Input profile: imgsz={IMAGE_SIZE} | conf={CONF_THRESHOLD:.2f} | "
            f"IoU={IOU_THRESHOLD:.2f} | max_det={MAX_DETECTIONS}"
        ),
        (
            f"Detections: {detections} | Mean Det Conf: {mean_conf:.3f} | "
            f"Detector latency: {detector_ms:.1f} ms | Throughput: {fps:.2f} FPS"
        ),
        (
            f"Step3 VisDrone validation: mAP50-95={ref['map50_95']:.5f} | "
            f"AP50={ref['ap50']:.5f} | Best epoch={ref['best_epoch']}"
        ),
        (
            f"Device: {DEVICE} | FP16: {USE_FP16} | "
            "Confidence is not ground-truth accuracy."
        ),
    ]

    y = 34

    for line in lines:
        cv2.putText(
            footer,
            line,
            (18, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.58,
            (245, 245, 245),
            1,
            cv2.LINE_AA,
        )
        y += 32

    return np.vstack([annotated, footer])


def maybe_transcode_h264(source_path: Path) -> Path:
    if not shutil.which("ffmpeg"):
        return source_path

    target = source_path.with_name(source_path.stem + "_h264.mp4")

    result = subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-loglevel",
            "error",
            "-i",
            str(source_path),
            "-c:v",
            "libx264",
            "-preset",
            "medium",
            "-crf",
            "20",
            "-pix_fmt",
            "yuv420p",
            "-an",
            str(target),
        ],
        check=False,
    )

    if result.returncode == 0 and target.exists():
        return target

    return source_path


## 5. BPD-YOLOn/L-FPN `DySample` compatibility

The project BPD checkpoint includes the custom `DySample` operator.  
It must be registered before the checkpoint is unpickled.


In [29]:
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F


class DySample(nn.Module):
    def __init__(
        self,
        channels=None,
        scale=2,
        max_offset=0.25,
    ):
        super().__init__()

        self.channels = int(channels) if channels is not None else None
        self.scale = int(scale)
        self.max_offset = float(max_offset)

        if self.channels is None:
            self.offset = nn.LazyConv2d(
                2,
                kernel_size=1,
                stride=1,
                padding=0,
            )
            self._zero_initialized = False
        else:
            self.offset = nn.Conv2d(
                self.channels,
                2,
                kernel_size=1,
                stride=1,
                padding=0,
            )

            with torch.no_grad():
                nn.init.zeros_(self.offset.weight)

                if self.offset.bias is not None:
                    nn.init.zeros_(self.offset.bias)

    def _materialize_legacy_lazy_offset(self, x):
        if (
            isinstance(self.offset, nn.LazyConv2d)
            and not getattr(self, "_zero_initialized", False)
        ):
            _ = self.offset(x)

            with torch.no_grad():
                nn.init.zeros_(self.offset.weight)

                if self.offset.bias is not None:
                    nn.init.zeros_(self.offset.bias)

            self._zero_initialized = True

    def forward(self, x):
        channels = getattr(self, "channels", None)

        if channels is not None and x.shape[1] != int(channels):
            raise RuntimeError(
                f"DySample expected {int(channels)} channels "
                f"but received {x.shape[1]}."
            )

        self._materialize_legacy_lazy_offset(x)

        b, _, h, w = x.shape
        scale = int(getattr(self, "scale", 2))
        max_offset = float(getattr(self, "max_offset", 0.25))

        oh = h * scale
        ow = w * scale

        raw = self.offset(x)

        raw = F.interpolate(
            raw,
            size=(oh, ow),
            mode="bilinear",
            align_corners=False,
        )

        raw = torch.tanh(raw) * max_offset

        ys = (
            (torch.arange(oh, device=x.device, dtype=x.dtype) + 0.5)
            / oh
            * 2.0
            - 1.0
        )

        xs = (
            (torch.arange(ow, device=x.device, dtype=x.dtype) + 0.5)
            / ow
            * 2.0
            - 1.0
        )

        yy, xx = torch.meshgrid(ys, xs, indexing="ij")

        base = (
            torch.stack((xx, yy), dim=-1)
            .unsqueeze(0)
            .expand(b, -1, -1, -1)
        )

        dx = raw[:, 0] * (2.0 / max(w, 1))
        dy = raw[:, 1] * (2.0 / max(h, 1))
        grid = base + torch.stack((dx, dy), dim=-1)

        return F.grid_sample(
            x,
            grid,
            mode="bilinear",
            padding_mode="border",
            align_corners=False,
        )


def register_bpd_compatibility():
    import ultralytics.nn.tasks as ultralytics_tasks

    setattr(sys.modules["__main__"], "DySample", DySample)
    setattr(ultralytics_tasks, "DySample", DySample)

    try:
        import ultralytics.nn.modules as ultralytics_modules
        setattr(ultralytics_modules, "DySample", DySample)
    except Exception:
        pass

    try:
        torch.serialization.add_safe_globals([DySample])
    except Exception:
        pass


register_bpd_compatibility()
print("BPD DySample compatibility: REGISTERED")


BPD DySample compatibility: REGISTERED


## 6. Common ByteTrack adapter used by all three detectors


In [30]:
from ultralytics.engine.results import Boxes
from ultralytics.trackers.byte_tracker import BYTETracker


class CommonByteTracker:
    def __init__(self):
        args = SimpleNamespace(
            tracker_type="bytetrack",
            track_high_thresh=TRACK_HIGH_THRESH,
            track_low_thresh=TRACK_LOW_THRESH,
            new_track_thresh=NEW_TRACK_THRESH,
            track_buffer=TRACK_BUFFER,
            match_thresh=MATCH_THRESH,
            fuse_score=FUSE_SCORE,
        )

        self.tracker = BYTETracker(args)

    def update(self, detections: DetectionBatch, frame):
        height, width = frame.shape[:2]

        if len(detections) == 0:
            data = torch.empty((0, 6), dtype=torch.float32)
        else:
            class_column = np.zeros(
                (len(detections), 1),
                dtype=np.float32,
            )

            data_np = np.concatenate(
                [
                    detections.boxes.astype(np.float32),
                    detections.scores.reshape(-1, 1).astype(np.float32),
                    class_column,
                ],
                axis=1,
            )

            data = torch.from_numpy(data_np)

        boxes = Boxes(data, (height, width))
        tracks = self.tracker.update(boxes, frame)

        if tracks is None or len(tracks) == 0:
            return (
                np.zeros((0, 4), dtype=np.float32),
                np.zeros((0,), dtype=np.int32),
                np.zeros((0,), dtype=np.float32),
            )

        tracks = np.asarray(tracks, dtype=np.float32)

        if tracks.ndim == 1:
            tracks = tracks[None, :]

        # Current ByteTrack output:
        # [x1, y1, x2, y2, track_id, score, cls, detection_index]
        return (
            tracks[:, :4],
            tracks[:, 4].astype(np.int32),
            tracks[:, 5].astype(np.float32),
        )


print("Common ByteTrack adapter: READY")


Common ByteTrack adapter: READY


## 7. Detector backends


In [31]:
from ultralytics import YOLO


class UltralyticsDetector:
    def __init__(
        self,
        model_path: Path,
        model_name: str,
        is_bpd: bool = False,
    ):
        self.model_path = Path(model_path)
        self.model_name = model_name

        if is_bpd:
            register_bpd_compatibility()

        self.model = YOLO(str(self.model_path))

        try:
            self.model.names = {0: "person"}
        except Exception:
            pass

    def predict(self, frame) -> DetectionBatch:
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.perf_counter()

        results = self.model.predict(
            source=frame,
            imgsz=IMAGE_SIZE,
            conf=CONF_THRESHOLD,
            iou=IOU_THRESHOLD,
            max_det=MAX_DETECTIONS,
            classes=[0],
            device=DEVICE,
            half=USE_FP16 and torch.cuda.is_available(),
            verbose=False,
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed_ms = (time.perf_counter() - start) * 1000.0
        result = results[0]

        if result.boxes is None or len(result.boxes) == 0:
            return DetectionBatch.empty(elapsed_ms)

        boxes = (
            result.boxes.xyxy
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        scores = (
            result.boxes.conf
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        classes = np.zeros((len(boxes),), dtype=np.float32)

        return DetectionBatch(
            boxes=boxes,
            scores=scores,
            classes=classes,
            detector_ms=elapsed_ms,
        )


class RTDETRR18Detector:
    def __init__(self, model_path: Path):
        self.model_path = Path(model_path)
        self.model_name = "RT-DETR-R18"

        self.local_root = Path("/content/aerial_rtdetr_inference")
        self.repo_root = self.local_root / "RT-DETR"
        self.rtdetr_root = self.repo_root / "rtdetrv2_pytorch"

        self._prepare_repository()
        self._build_model()

    def _prepare_repository(self):
        self.local_root.mkdir(parents=True, exist_ok=True)

        if not self.repo_root.exists():
            subprocess.check_call(
                [
                    "git",
                    "clone",
                    "--filter=blob:none",
                    RTDETR_REPO_URL,
                    str(self.repo_root),
                ]
            )

        subprocess.check_call(
            [
                "git",
                "-C",
                str(self.repo_root),
                "fetch",
                "--all",
                "--tags",
                "--prune",
            ]
        )

        subprocess.check_call(
            [
                "git",
                "-C",
                str(self.repo_root),
                "checkout",
                "--force",
                RTDETR_REPO_COMMIT,
            ]
        )

        if not self.rtdetr_root.exists():
            raise FileNotFoundError(self.rtdetr_root)

    def _build_model(self):
        if str(self.rtdetr_root) not in sys.path:
            sys.path.insert(0, str(self.rtdetr_root))

        from src.core import YAMLConfig

        config_dir = self.rtdetr_root / "configs" / "custom"
        config_dir.mkdir(parents=True, exist_ok=True)

        config_path = (
            config_dir
            / "rtdetr_r18_visdrone_person_inference.yml"
        )

        config_text = (
            "__include__:\n"
            "  - ../rtdetr/rtdetr_r18vd_6x_coco.yml\n"
            "\n"
            "num_classes: 1\n"
            "remap_mscoco_category: False\n"
            f"eval_spatial_size: [{IMAGE_SIZE}, {IMAGE_SIZE}]\n"
            "\n"
            "PResNet:\n"
            "  pretrained: False\n"
        )

        config_path.write_text(
            config_text,
            encoding="utf-8",
        )

        old_cwd = Path.cwd()

        try:
            os.chdir(self.rtdetr_root)

            cfg = YAMLConfig(
                str(config_path),
                device=DEVICE,
                use_amp=USE_FP16 and torch.cuda.is_available(),
            )
        finally:
            os.chdir(old_cwd)

        checkpoint = torch.load(
            self.model_path,
            map_location="cpu",
            weights_only=False,
        )

        state = None
        ema = checkpoint.get("ema")

        if ema is not None:
            if isinstance(ema, dict) and "module" in ema:
                state = ema["module"]
            elif isinstance(ema, dict):
                state = ema

        if state is None:
            state = checkpoint.get("model")

        if state is None:
            raise RuntimeError(
                "Could not find RT-DETR model weights in checkpoint."
            )

        cfg.model.load_state_dict(state, strict=True)

        self.model = cfg.model.deploy().to(DEVICE).eval()
        self.postprocessor = cfg.postprocessor.deploy()

        del checkpoint
        del state
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def _preprocess(self, frame):
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        resized = cv2.resize(
            rgb,
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=cv2.INTER_LINEAR,
        )

        array = resized.astype(np.float32) / 255.0

        tensor = (
            torch.from_numpy(array)
            .permute(2, 0, 1)
            .unsqueeze(0)
            .contiguous()
        )

        return tensor.to(DEVICE, non_blocking=True)

    def predict(self, frame) -> DetectionBatch:
        height, width = frame.shape[:2]
        tensor = self._preprocess(frame)

        # RT-DETR/DETR target sizes use [height, width].
        original_size = torch.tensor(
            [[height, width]],
            dtype=torch.float32,
            device=DEVICE,
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.perf_counter()

        with torch.inference_mode():
            if USE_FP16 and torch.cuda.is_available():
                autocast_context = torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                )
            else:
                autocast_context = contextlib.nullcontext()

            with autocast_context:
                outputs = self.model(tensor)

                labels, boxes, scores = self.postprocessor(
                    outputs,
                    original_size,
                )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed_ms = (time.perf_counter() - start) * 1000.0

        labels = labels[0].detach().cpu().numpy()
        boxes = boxes[0].detach().cpu().numpy().astype(np.float32)
        scores = scores[0].detach().cpu().numpy().astype(np.float32)

        mask = (scores >= CONF_THRESHOLD) & (labels == 0)

        boxes = boxes[mask]
        scores = scores[mask]

        if len(scores) > MAX_DETECTIONS:
            order = np.argsort(-scores)[:MAX_DETECTIONS]
            boxes = boxes[order]
            scores = scores[order]

        if len(boxes) == 0:
            return DetectionBatch.empty(elapsed_ms)

        classes = np.zeros((len(boxes),), dtype=np.float32)

        return DetectionBatch(
            boxes=boxes,
            scores=scores,
            classes=classes,
            detector_ms=elapsed_ms,
        )


def build_detector(model_name: str):
    if model_name == "YOLO26s":
        return UltralyticsDetector(
            YOLO26S_MODEL_PATH,
            model_name="YOLO26s",
            is_bpd=False,
        )

    if model_name == "RT-DETR-R18":
        return RTDETRR18Detector(RTDETR_MODEL_PATH)

    if model_name == "BPD-YOLOn/L-FPN":
        return UltralyticsDetector(
            BPD_MODEL_PATH,
            model_name="BPD-YOLOn/L-FPN",
            is_bpd=True,
        )

    raise ValueError(f"Unsupported model: {model_name}")


print("Detector backends: READY")


Detector backends: READY


## 8. Preflight paths and selected media


In [32]:
MODEL_PATHS = {
    "YOLO26s": YOLO26S_MODEL_PATH,
    "RT-DETR-R18": RTDETR_MODEL_PATH,
    "BPD-YOLOn/L-FPN": BPD_MODEL_PATH,
}

VALID_RUN_MODELS = {
    "YOLO26s",
    "RT-DETR-R18",
    "BPD-YOLOn/L-FPN",
    "ALL",
}

if RUN_MODEL not in VALID_RUN_MODELS:
    raise ValueError(
        f"RUN_MODEL must be one of: {sorted(VALID_RUN_MODELS)}"
    )

if not MEDIA_PATH.exists():
    raise FileNotFoundError(
        f"Media file was not found: {MEDIA_PATH}"
    )

models_to_run = (
    ["YOLO26s", "RT-DETR-R18", "BPD-YOLOn/L-FPN"]
    if RUN_MODEL == "ALL"
    else [RUN_MODEL]
)

for model_name in models_to_run:
    model_path = MODEL_PATHS[model_name]

    if not model_path.exists():
        raise FileNotFoundError(
            f"{model_name} checkpoint was not found: {model_path}"
        )

    print(f"{model_name:18s} -> {model_path}")

media_type = classify_media(MEDIA_PATH)

print("\nMedia type :", media_type)
print("Models     :", models_to_run)
print("Output root:", OUTPUT_ROOT)
print("\nPRE-FLIGHT: PASSED")


FileNotFoundError: Media file was not found: https:/drive.google.com/drive/folders/1Gv5E3ju5fg2nC9C2GNg7IHPeNbPsuuKD?usp=drive_link

## 9. Warm-up and image/video processing functions


In [ ]:
def warmup_detector(detector, frame):
    if WARMUP_RUNS <= 0:
        return

    print(
        f"Warming up {detector.model_name} "
        f"for {WARMUP_RUNS} run(s)..."
    )

    for _ in range(WARMUP_RUNS):
        _ = detector.predict(frame)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print("Warm-up: COMPLETE")


def process_image(
    model_name: str,
    detector,
    media_path: Path,
):
    image = cv2.imread(str(media_path))

    if image is None:
        raise RuntimeError(
            f"OpenCV could not read image: {media_path}"
        )

    warmup_detector(detector, image)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    detections = detector.predict(image)

    mean_conf = (
        float(detections.scores.mean())
        if len(detections)
        else 0.0
    )

    annotated = draw_detection_boxes(
        image,
        detections.boxes,
        detections.scores,
    )

    canvas = build_image_footer(
        annotated=annotated,
        model_name=model_name,
        model_path=MODEL_PATHS[model_name],
        detector_ms=detections.detector_ms,
        detections=len(detections),
        mean_conf=mean_conf,
    )

    model_slug = sanitize_name(model_name)
    stem = media_path.stem

    image_output = (
        OUTPUT_ROOT
        / f"{stem}__{model_slug}__detected.jpg"
    )

    csv_output = (
        OUTPUT_ROOT
        / f"{stem}__{model_slug}__detections.csv"
    )

    json_output = (
        OUTPUT_ROOT
        / f"{stem}__{model_slug}__summary.json"
    )

    cv2.imwrite(
        str(image_output),
        canvas,
        [int(cv2.IMWRITE_JPEG_QUALITY), 95],
    )

    rows = []

    for index, (box, score) in enumerate(
        zip(detections.boxes, detections.scores),
        start=1,
    ):
        x1, y1, x2, y2 = [float(v) for v in box]

        rows.append(
            {
                "detection_id": index,
                "class": "person",
                "confidence": float(score),
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "width": x2 - x1,
                "height": y2 - y1,
            }
        )

    if SAVE_CSV:
        pd.DataFrame(rows).to_csv(
            csv_output,
            index=False,
        )

    max_gpu_mb = (
        torch.cuda.max_memory_allocated() / 1024 / 1024
        if torch.cuda.is_available()
        else 0.0
    )

    summary = {
        "media_type": "image",
        "source": str(media_path),
        "model": model_name,
        "checkpoint": str(MODEL_PATHS[model_name]),
        "checkpoint_sha256": sha256_file(
            MODEL_PATHS[model_name]
        ),
        "device": DEVICE,
        "fp16": USE_FP16,
        "imgsz": IMAGE_SIZE,
        "conf_threshold": CONF_THRESHOLD,
        "iou_threshold": IOU_THRESHOLD,
        "max_detections": MAX_DETECTIONS,
        "detections": len(detections),
        "mean_detection_confidence": mean_conf,
        "detector_latency_ms": detections.detector_ms,
        "detector_throughput_fps": (
            1000.0 / detections.detector_ms
            if detections.detector_ms > 0
            else 0.0
        ),
        "max_torch_gpu_memory_mb": max_gpu_mb,
        "step3_validation": STEP3_VALIDATION[model_name],
        "note": (
            "Detection confidence and throughput are measured on this media. "
            "Step3 validation metrics are reference VisDrone metrics, not "
            "ground-truth accuracy for this image."
        ),
        "output_image": str(image_output),
    }

    if SAVE_JSON:
        json_output.write_text(
            json.dumps(summary, indent=2),
            encoding="utf-8",
        )

    print("\n" + "=" * 80)
    print(f"IMAGE RESULT — {model_name}")
    print("=" * 80)
    print("Detections       :", len(detections))
    print("Mean confidence  :", f"{mean_conf:.4f}")
    print(
        "Detector latency :",
        f"{detections.detector_ms:.2f} ms",
    )
    print("Output image     :", image_output)
    print("=" * 80)

    rgb = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)
    display(Image.fromarray(rgb))

    return summary


def process_video(
    model_name: str,
    detector,
    media_path: Path,
):
    cap = cv2.VideoCapture(str(media_path))

    if not cap.isOpened():
        raise RuntimeError(
            f"OpenCV could not open video: {media_path}"
        )

    source_fps = float(cap.get(cv2.CAP_PROP_FPS))

    if not np.isfinite(source_fps) or source_fps <= 0:
        source_fps = 25.0

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    first_ok, first_frame = cap.read()

    if not first_ok:
        cap.release()
        raise RuntimeError(
            "The video contains no readable frames."
        )

    warmup_detector(detector, first_frame)
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    tracker = CommonByteTracker()

    model_slug = sanitize_name(model_name)
    stem = media_path.stem

    raw_output = (
        OUTPUT_ROOT
        / f"{stem}__{model_slug}__ByteTrack_raw.mp4"
    )

    csv_output = (
        OUTPUT_ROOT
        / f"{stem}__{model_slug}__tracks.csv"
    )

    json_output = (
        OUTPUT_ROOT
        / f"{stem}__{model_slug}__summary.json"
    )

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        str(raw_output),
        fourcc,
        source_fps,
        (width, height),
    )

    if not writer.isOpened():
        cap.release()
        raise RuntimeError(
            f"Could not create output video: {raw_output}"
        )

    trails = defaultdict(
        lambda: deque(maxlen=TRAIL_LENGTH)
    )

    unique_track_ids = set()
    csv_rows = []

    processing_fps_history = deque(
        maxlen=FPS_SMOOTHING_WINDOW
    )

    detector_latency_history = []
    total_latency_history = []
    detection_confidences = []

    processed_frames = 0
    skipped_frames = 0

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    overall_start = time.perf_counter()

    progress = tqdm(
        total=total_frames if total_frames > 0 else None,
        desc=f"{model_name} video",
        unit="frame",
    )

    frame_index = 0

    try:
        while True:
            ok, frame = cap.read()

            if not ok:
                break

            frame_index += 1

            if (
                VIDEO_FRAME_STRIDE > 1
                and (frame_index - 1) % VIDEO_FRAME_STRIDE != 0
            ):
                skipped_frames += 1
                progress.update(1)
                continue

            frame_start = time.perf_counter()

            detections = detector.predict(frame)

            (
                track_boxes,
                track_ids,
                track_scores,
            ) = tracker.update(
                detections,
                frame,
            )

            for box, track_id, score in zip(
                track_boxes,
                track_ids,
                track_scores,
            ):
                x1, y1, x2, y2 = [float(v) for v in box]

                center = (
                    int(round((x1 + x2) / 2)),
                    int(round((y1 + y2) / 2)),
                )

                trails[int(track_id)].append(center)
                unique_track_ids.add(int(track_id))

                csv_rows.append(
                    {
                        "frame": frame_index,
                        "time_seconds": (
                            (frame_index - 1) / source_fps
                        ),
                        "track_id": int(track_id),
                        "class": "person",
                        "confidence": float(score),
                        "x1": x1,
                        "y1": y1,
                        "x2": x2,
                        "y2": y2,
                        "width": x2 - x1,
                        "height": y2 - y1,
                    }
                )

            if len(detections):
                detection_confidences.extend(
                    detections.scores.tolist()
                )

            annotated = draw_detection_boxes(
                frame=frame,
                boxes=track_boxes,
                scores=track_scores,
                track_ids=track_ids,
                trails=trails,
            )

            elapsed_before_overlay = (
                time.perf_counter() - frame_start
            )

            frame_processing_fps = (
                1.0 / elapsed_before_overlay
                if elapsed_before_overlay > 0
                else 0.0
            )

            processing_fps_history.append(
                frame_processing_fps
            )

            smooth_fps = float(
                np.mean(processing_fps_history)
            )

            total_ms = elapsed_before_overlay * 1000.0

            mean_frame_conf = (
                float(detections.scores.mean())
                if len(detections)
                else 0.0
            )

            annotated = add_video_status_overlay(
                frame=annotated,
                model_name=model_name,
                frame_number=frame_index,
                source_fps=source_fps,
                processing_fps=smooth_fps,
                detector_ms=detections.detector_ms,
                total_ms=total_ms,
                detections=len(detections),
                active_tracks=len(track_ids),
                unique_tracks=len(unique_track_ids),
                mean_conf=mean_frame_conf,
            )

            writer.write(annotated)

            detector_latency_history.append(
                detections.detector_ms
            )

            total_latency_history.append(total_ms)
            processed_frames += 1

            progress.update(1)

            progress.set_postfix(
                {
                    "FPS": f"{smooth_fps:.1f}",
                    "IDs": len(unique_track_ids),
                    "det": len(detections),
                },
                refresh=False,
            )

    finally:
        progress.close()
        cap.release()
        writer.release()

    overall_seconds = time.perf_counter() - overall_start

    if SAVE_CSV:
        pd.DataFrame(csv_rows).to_csv(
            csv_output,
            index=False,
        )

    final_video = (
        maybe_transcode_h264(raw_output)
        if SAVE_VIDEO
        else raw_output
    )

    if final_video != raw_output and raw_output.exists():
        try:
            raw_output.unlink()
        except Exception:
            pass

    avg_detector_ms = (
        float(np.mean(detector_latency_history))
        if detector_latency_history
        else 0.0
    )

    median_detector_ms = (
        float(np.median(detector_latency_history))
        if detector_latency_history
        else 0.0
    )

    avg_total_ms = (
        float(np.mean(total_latency_history))
        if total_latency_history
        else 0.0
    )

    measured_processing_fps = (
        processed_frames / overall_seconds
        if overall_seconds > 0
        else 0.0
    )

    mean_conf = (
        float(np.mean(detection_confidences))
        if detection_confidences
        else 0.0
    )

    max_gpu_mb = (
        torch.cuda.max_memory_allocated() / 1024 / 1024
        if torch.cuda.is_available()
        else 0.0
    )

    source_duration = (
        total_frames / source_fps
        if total_frames > 0
        else None
    )

    summary = {
        "media_type": "video",
        "source": str(media_path),
        "model": model_name,
        "checkpoint": str(MODEL_PATHS[model_name]),
        "checkpoint_sha256": sha256_file(
            MODEL_PATHS[model_name]
        ),
        "device": DEVICE,
        "fp16": USE_FP16,
        "imgsz": IMAGE_SIZE,
        "conf_threshold": CONF_THRESHOLD,
        "iou_threshold": IOU_THRESHOLD,
        "max_detections": MAX_DETECTIONS,
        "bytetrack": {
            "track_high_thresh": TRACK_HIGH_THRESH,
            "track_low_thresh": TRACK_LOW_THRESH,
            "new_track_thresh": NEW_TRACK_THRESH,
            "track_buffer": TRACK_BUFFER,
            "match_thresh": MATCH_THRESH,
            "fuse_score": FUSE_SCORE,
            "trail_length": TRAIL_LENGTH,
        },
        "source_video": {
            "fps": source_fps,
            "frames": total_frames,
            "width": width,
            "height": height,
            "duration_seconds": source_duration,
        },
        "runtime": {
            "processed_frames": processed_frames,
            "skipped_frames": skipped_frames,
            "overall_processing_seconds": overall_seconds,
            "effective_processing_fps": measured_processing_fps,
            "average_detector_latency_ms": avg_detector_ms,
            "median_detector_latency_ms": median_detector_ms,
            "average_total_frame_latency_ms": avg_total_ms,
            "max_torch_gpu_memory_mb": max_gpu_mb,
        },
        "detections_and_tracking": {
            "track_rows": len(csv_rows),
            "unique_track_ids": len(unique_track_ids),
            "mean_detection_confidence": mean_conf,
        },
        "step3_validation": STEP3_VALIDATION[model_name],
        "note": (
            "Runtime FPS, latency and confidence are measured on this video. "
            "Step3 mAP/AP50 are reference VisDrone validation metrics. "
            "HOTA, IDF1, MOTA and ID-switch accuracy require annotated "
            "tracking ground truth."
        ),
        "output_video": str(final_video),
        "output_csv": str(csv_output) if SAVE_CSV else None,
    }

    if SAVE_JSON:
        json_output.write_text(
            json.dumps(summary, indent=2),
            encoding="utf-8",
        )

    print("\n" + "=" * 88)
    print(f"VIDEO RESULT — {model_name}")
    print("=" * 88)
    print("Processed frames        :", processed_frames)
    print("Source FPS              :", f"{source_fps:.3f}")
    print(
        "Effective processing FPS:",
        f"{measured_processing_fps:.3f}",
    )
    print(
        "Average detector latency:",
        f"{avg_detector_ms:.3f} ms",
    )
    print(
        "Average total latency   :",
        f"{avg_total_ms:.3f} ms",
    )
    print("Unique track IDs        :", len(unique_track_ids))
    print("Mean detection confidence:", f"{mean_conf:.4f}")
    print("Output video            :", final_video)
    print("CSV                     :", csv_output)
    print("JSON                    :", json_output)
    print("=" * 88)

    if final_video.exists():
        display(Video(str(final_video), embed=False))

    return summary


## 10. Run the selected model(s)


In [ ]:
all_summaries = []

for model_name in models_to_run:
    print("\n" + "#" * 96)
    print(f"LOADING MODEL: {model_name}")
    print("#" * 96)

    detector = build_detector(model_name)
    print(f"{model_name} loaded successfully.")

    if media_type == "image":
        summary = process_image(
            model_name=model_name,
            detector=detector,
            media_path=MEDIA_PATH,
        )
    else:
        summary = process_video(
            model_name=model_name,
            detector=detector,
            media_path=MEDIA_PATH,
        )

    all_summaries.append(summary)

    del detector
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


combined_summary_path = (
    OUTPUT_ROOT
    / f"{MEDIA_PATH.stem}__all_run_summaries.json"
)

combined_summary_path.write_text(
    json.dumps(all_summaries, indent=2),
    encoding="utf-8",
)

print("\nAll requested processing is complete.")
print("Combined summary:", combined_summary_path)


## 11. Runtime comparison table


In [ ]:
comparison_rows = []

for summary in all_summaries:
    row = {
        "model": summary["model"],
        "step3_val_map50_95": (
            summary["step3_validation"]["map50_95"]
        ),
        "step3_val_ap50": (
            summary["step3_validation"]["ap50"]
        ),
    }

    if summary["media_type"] == "video":
        row.update(
            {
                "processing_fps": (
                    summary["runtime"]["effective_processing_fps"]
                ),
                "detector_latency_ms": (
                    summary["runtime"]["average_detector_latency_ms"]
                ),
                "total_frame_latency_ms": (
                    summary["runtime"]["average_total_frame_latency_ms"]
                ),
                "unique_track_ids": (
                    summary["detections_and_tracking"]["unique_track_ids"]
                ),
                "mean_detection_confidence": (
                    summary["detections_and_tracking"][
                        "mean_detection_confidence"
                    ]
                ),
            }
        )
    else:
        row.update(
            {
                "processing_fps": (
                    summary["detector_throughput_fps"]
                ),
                "detector_latency_ms": (
                    summary["detector_latency_ms"]
                ),
                "total_frame_latency_ms": np.nan,
                "unique_track_ids": np.nan,
                "mean_detection_confidence": (
                    summary["mean_detection_confidence"]
                ),
            }
        )

    comparison_rows.append(row)


comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

comparison_csv = (
    OUTPUT_ROOT
    / f"{MEDIA_PATH.stem}__runtime_comparison.csv"
)

comparison_df.to_csv(
    comparison_csv,
    index=False,
)

print("Runtime comparison CSV:", comparison_csv)


## 12. Interpretation notes

- **Image mode never creates track IDs.**
- **Video mode uses one common ByteTrack adapter for all three detectors.**
- The annotated video reports measured FPS and latency from the current Colab runtime.
- The fixed Step-3 `mAP50-95` and `AP50` values are included only as reference metrics from the controlled VisDrone validation experiment.
- A real accuracy score for an arbitrary new image/video requires ground-truth annotations.
- Formal tracking metrics such as HOTA, IDF1, MOTA and ID switches require a tracking ground-truth file.
- For Step 4, repeat the workload using ONNX/TensorRT on the target RTX 3070 and compare accuracy retention, latency, FPS and VRAM.
